# La Digithèque — Préparation du RAG hybride CECRL + HSK

**Version initiale :** préparation des corpus, extraction, normalisation prudente, embeddings et index hybride BM25 + FAISS.

Ce notebook est conçu pour Google Colab. Il prépare un pack exportable que le Service IA FastAPI pourra ensuite charger.

## Périmètre de cette version
- CECRL : fichiers JSON.
- HSK : fichiers PDF.
- Audios HSK : volontairement exclus de cette première version.
- Modèle d'embeddings candidat : `intfloat/multilingual-e5-base`.
- Recherche : BM25 + recherche vectorielle, fusion Reciprocal Rank Fusion (RRF).

> Important : le notebook commence par inventorier et extraire les sources. Les JSON peuvent avoir des structures hétérogènes et certains PDF peuvent être scannés. Les contenus insuffisamment extraits sont signalés au lieu d'être inventés. Vérifie les résultats avant de construire l'index définitif.

## 0. Configuration de l'environnement

Exécuter cette cellule dans un runtime Colab avec Python 3.10 ou 3.11 de préférence. Un runtime GPU est utile pour accélérer les embeddings, mais le notebook peut aussi fonctionner sur CPU avec davantage de temps.

In [ ]:
!pip -q install pymupdf sentence-transformers faiss-cpu rank-bm25 pandas tqdm

In [ ]:
from pathlib import Path
import os, re, json, csv, hashlib, zipfile, shutil, textwrap, warnings
from collections import Counter
import pandas as pd
from tqdm.auto import tqdm

BASE_DIR = Path("/content/langue_ia_rag")
UPLOAD_DIR = BASE_DIR / "uploads"
EXTRACT_DIR = BASE_DIR / "sources"
WORK_DIR = BASE_DIR / "work"
OUTPUT_DIR = BASE_DIR / "rag_export"

for p in [UPLOAD_DIR, EXTRACT_DIR, WORK_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Répertoires prêts :", BASE_DIR)

## 1. Importer les archives ZIP

Prépare idéalement deux archives :
- `CECRL.zip` : les fichiers JSON CECRL.
- `HSK.zip` : les PDF HSK uniquement.

Tu peux aussi importer plusieurs archives. Évite d'inclure les MP3 dans cette première version pour réduire le transfert et le traitement.

In [ ]:
from google.colab import files

uploaded = files.upload()
for filename, content in uploaded.items():
    target = UPLOAD_DIR / Path(filename).name
    target.write_bytes(content)
    print(f"Importé : {target.name} ({target.stat().st_size / (1024**2):.1f} Mo)")

## 2. Décompresser les ZIP de manière sécurisée

La fonction suivante bloque les chemins malveillants contenus dans une archive ZIP (path traversal). Les fichiers sont extraits dans un sous-dossier portant le nom de l'archive.

In [ ]:
def safe_extract_zip(zip_path: Path, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            name = member.filename.replace("\\", "/")
            if name.startswith("/") or re.match(r"^[A-Za-z]:", name):
                raise ValueError(f"Chemin absolu refusé dans {zip_path.name}: {name}")
            dest = (destination / name).resolve()
            if root not in dest.parents and dest != root:
                raise ValueError(f"Chemin hors du dossier refusé : {name}")
        zf.extractall(destination)
    return destination

archives = sorted(UPLOAD_DIR.glob("*.zip"))
if not archives:
    print("Aucune archive ZIP trouvée. Importez vos ZIP puis relancez cette cellule.")
else:
    for archive in archives:
        folder_name = archive.stem.lower()
        safe_extract_zip(archive, EXTRACT_DIR / folder_name)
        print("Extrait :", archive.name, "→", EXTRACT_DIR / folder_name)

## 3. Inventaire des fichiers extraits

Cette étape vérifie la présence des JSON et PDF, leur taille et leur répartition. Elle ne modifie pas les sources.

In [ ]:
all_files = [p for p in EXTRACT_DIR.rglob("*") if p.is_file()]
inventory = []
for p in all_files:
    inventory.append({
        "chemin_relatif": str(p.relative_to(EXTRACT_DIR)),
        "extension": p.suffix.lower(),
        "taille_Mo": round(p.stat().st_size / (1024**2), 3),
    })
inventory_df = pd.DataFrame(inventory)
display(inventory_df.groupby("extension").agg(
    nombre=("chemin_relatif", "count"),
    taille_totale_Mo=("taille_Mo", "sum")
).sort_values("nombre", ascending=False))
display(inventory_df.head(30))

## 4. Audit des JSON CECRL

Les JSON ont des schémas qui peuvent varier selon les langues et niveaux. Cette cellule :
- vérifie l'encodage et la validité JSON ;
- affiche le type racine et les clés principales ;
- compte les éléments et les champs textuels ;
- ne suppose pas un schéma unique.

In [ ]:
json_files = sorted(EXTRACT_DIR.rglob("*.json"))
json_audit = []

def load_json_file(path):
    # UTF-8 standard, avec tolérance BOM.
    with open(path, "r", encoding="utf-8-sig") as f:
        return json.load(f)

for p in json_files:
    row = {"fichier": str(p.relative_to(EXTRACT_DIR)), "valide": False,
           "type_racine": None, "cles_racine": "", "taille_octets": p.stat().st_size,
           "erreur": ""}
    try:
        obj = load_json_file(p)
        row["valide"] = True
        row["type_racine"] = type(obj).__name__
        if isinstance(obj, dict):
            row["cles_racine"] = ", ".join(list(obj.keys())[:30])
        elif isinstance(obj, list) and obj and isinstance(obj[0], dict):
            row["cles_racine"] = "élément[0]: " + ", ".join(list(obj[0].keys())[:30])
    except Exception as e:
        row["erreur"] = str(e)[:300]
    json_audit.append(row)

json_audit_df = pd.DataFrame(json_audit)
display(json_audit_df)
print("JSON valides :", int(json_audit_df["valide"].sum()), "/", len(json_audit_df))

## 5. Audit et extraction textuelle des PDF HSK

Chaque page PDF est extraite séparément. Les pages avec très peu de texte sont marquées comme potentiellement scannées ou nécessitant un OCR.

Cette version ne lance pas automatiquement d'OCR : il vaut mieux inspecter le taux de pages vides avant de choisir une solution OCR adaptée.

In [ ]:
import fitz  # PyMuPDF

pdf_files = sorted(EXTRACT_DIR.rglob("*.pdf"))
pdf_pages = []
pdf_audit = []

for pdf_path in tqdm(pdf_files, desc="Extraction PDF"):
    doc = fitz.open(pdf_path)
    extracted_chars = 0
    low_text_pages = 0
    for page_no, page in enumerate(doc, start=1):
        txt = page.get_text("text") or ""
        txt = re.sub(r"\\s+", " ", txt).strip()
        extracted_chars += len(txt)
        if len(txt) < 80:
            low_text_pages += 1
        pdf_pages.append({
            "source_file": str(pdf_path.relative_to(EXTRACT_DIR)),
            "page": page_no,
            "text": txt,
            "char_count": len(txt),
            "needs_ocr_review": len(txt) < 80,
            "referentiel": "HSK",
            "level": None,
            "language": "zh",
        })
    pdf_audit.append({
        "fichier": str(pdf_path.relative_to(EXTRACT_DIR)),
        "pages": len(doc),
        "caracteres_extraits": extracted_chars,
        "pages_faible_texte": low_text_pages,
        "pct_pages_faible_texte": round(100 * low_text_pages / max(len(doc), 1), 1),
    })
    doc.close()

pdf_audit_df = pd.DataFrame(pdf_audit)
display(pdf_audit_df)
print("Pages extraites :", len(pdf_pages))

## 6. Normalisation prudente des documents

On transforme les contenus en une structure commune minimale :
- `doc_id`
- `referentiel` (`CECRL` ou `HSK`)
- `language`
- `level` si détectable
- `text`
- `source_file`
- `source_locator` (JSON pointer simplifié ou numéro de page)
- `metadata`

Pour les JSON, le code conserve les objets contenant des champs textuels directs. Il n'invente pas de niveau ni de langue : lorsque ces informations ne sont pas détectées, elles restent `unknown` et devront être corrigées dans la configuration.

In [ ]:
def infer_language(path_text):
    s = path_text.lower()
    aliases = {
        "english": "en", "anglais": "en",
        "french": "fr", "francais": "fr", "français": "fr",
        "german": "de", "allemand": "de",
        "italian": "it", "italien": "it",
        "chinese": "zh", "mandarin": "zh", "hsk": "zh",
    }
    for key, value in aliases.items():
        if key in s:
            return value
    return "unknown"

def infer_level(path_text, referentiel):
    s = path_text.upper()
    if referentiel == "CECRL":
        m = re.search(r"(?<![A-Z0-9])(A1|A2|B1|B2|C1|C2)(?![A-Z0-9])", s)
        return m.group(1) if m else None
    m = re.search(r"HSK[ _-]?(4A|4B|[1-6])", s)
    return f"HSK{m.group(1)}" if m else None

TEXT_KEYS = {
    "question", "prompt", "text", "content", "statement", "description",
    "answer", "response", "explanation", "instruction", "title",
    "question_text", "correct_answer", "vocabulary", "word", "sentence",
    "definition", "grammar", "example", "translation"
}

def direct_text_values(obj):
    vals = []
    if not isinstance(obj, dict):
        return vals
    for k, v in obj.items():
        if isinstance(v, str) and (k.lower() in TEXT_KEYS or len(v.strip()) >= 80):
            cleaned = re.sub(r"\\s+", " ", v).strip()
            if cleaned:
                vals.append(f"{k}: {cleaned}")
    return vals

def walk_json_records(obj, path="root"):
    # Parcourt listes et dictionnaires sans sérialiser en double tout l'arbre.
    if isinstance(obj, list):
        for i, item in enumerate(obj):
            yield from walk_json_records(item, f"{path}[{i}]")
    elif isinstance(obj, dict):
        vals = direct_text_values(obj)
        if vals:
            yield path, "\\n".join(vals), obj
        for k, v in obj.items():
            if isinstance(v, (dict, list)):
                yield from walk_json_records(v, f"{path}.{k}")

documents = []

for jp in json_files:
    try:
        data = load_json_file(jp)
    except Exception:
        continue
    rel = str(jp.relative_to(EXTRACT_DIR))
    lang = infer_language(rel)
    level = infer_level(rel, "CECRL")
    for locator, txt, original in walk_json_records(data):
        if len(txt) < 20:
            continue
        documents.append({
            "referentiel": "CECRL",
            "language": lang,
            "level": level,
            "text": txt,
            "source_file": rel,
            "source_locator": locator,
            "metadata": {"json_keys": list(original.keys())[:40]},
        })

for page in pdf_pages:
    if page["char_count"] >= 80:
        documents.append({
            "referentiel": "HSK",
            "language": page["language"],
            "level": infer_level(page["source_file"], "HSK"),
            "text": page["text"],
            "source_file": page["source_file"],
            "source_locator": f"page:{page['page']}",
            "metadata": {"page": page["page"]},
        })

# Déduplication exacte du texte au sein d'une même source.
seen = set()
deduped = []
for d in documents:
    key = (d["referentiel"], d["source_file"], hashlib.sha256(d["text"].encode("utf-8")).hexdigest())
    if key not in seen:
        seen.add(key)
        deduped.append(d)
documents = deduped

print("Documents normalisés :", len(documents))
display(pd.DataFrame([{
    "referentiel": d["referentiel"], "language": d["language"],
    "level": d["level"], "source_file": d["source_file"],
    "source_locator": d["source_locator"], "longueur": len(d["text"])
} for d in documents]).head(20))

## 7. Contrôle qualité avant embeddings

Inspecter les statistiques et quelques exemples. Si les langues ou niveaux sont `unknown` ou absents alors qu'ils devraient être connus, corriger `infer_language` / `infer_level` avant de continuer.

Ne pas construire l'index tant que les exemples ne sont pas pédagogiquement exploitables.

In [ ]:
if not documents:
    raise RuntimeError("Aucun document exploitable. Vérifiez les ZIP, JSON et PDF.")

docs_df = pd.DataFrame(documents)
display(docs_df.groupby(["referentiel", "language", "level"], dropna=False).size().reset_index(name="documents"))
for i, d in enumerate(documents[:8], start=1):
    print(f"--- Exemple {i} | {d['referentiel']} | {d['source_file']} | {d['source_locator']} ---")
    print(d["text"][:700], "\\n")

## 8. Découpage en chunks

Les chunks sont construits à partir des documents normalisés. Les frontières de page/source sont conservées dans les métadonnées. Les tailles ci-dessous sont en caractères, donc à ajuster après inspection de la langue et du type de contenu.

In [ ]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

def split_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = re.sub(r"\\s+", " ", text).strip()
    if len(text) <= chunk_size:
        return [text] if text else []
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            # Essaie de couper à une frontière de phrase ou d'espace.
            boundary = max(text.rfind(". ", start, end), text.rfind("。", start, end), text.rfind(" ", start, end))
            if boundary > start + int(chunk_size * 0.6):
                end = boundary + 1
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

chunks = []
for d in documents:
    for idx, chunk in enumerate(split_text(d["text"])):
        if len(chunk) < 30:
            continue
        chunks.append({
            "chunk_id": hashlib.sha1(
                f"{d['source_file']}|{d['source_locator']}|{idx}|{chunk}".encode("utf-8")
            ).hexdigest(),
            "referentiel": d["referentiel"],
            "language": d["language"],
            "level": d["level"],
            "text": chunk,
            "source_file": d["source_file"],
            "source_locator": d["source_locator"],
            "chunk_index": idx,
            "metadata": d["metadata"],
        })

print("Nombre de chunks :", len(chunks))

## 9. Générer les embeddings multilingues

Modèle candidat : `intfloat/multilingual-e5-base`.

E5 utilise des préfixes :
- documents : `passage: ...`
- requêtes : `query: ...`

Il faut conserver le nom exact du modèle et les paramètres d'encodage dans le pack exporté. Le même modèle doit encoder les requêtes au moment de l'utilisation dans FastAPI.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device)

model = SentenceTransformer(EMBEDDING_MODEL, device=device)

passages = ["passage: " + c["text"] for c in chunks]
embeddings = model.encode(
    passages,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print("Shape embeddings :", embeddings.shape)

## 10. Construire les index vectoriel et BM25

FAISS sert à la recherche vectorielle. BM25 sert à la recherche lexicale. La fusion RRF combine les classements sans supposer que les scores BM25 et vectoriels sont sur la même échelle.

In [ ]:
import faiss
from rank_bm25 import BM25Okapi
import pickle
import numpy as np

vector_matrix = np.asarray(embeddings, dtype="float32")
faiss.normalize_L2(vector_matrix)

dimension = vector_matrix.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(vector_matrix)

def tokenize(text):
    # Tokenisation simple multilingue, à améliorer si l'évaluation le justifie.
    return re.findall(r"[\\w]+|[\\u4e00-\\u9fff]", text.lower(), flags=re.UNICODE)

bm25_corpus = [tokenize(c["text"]) for c in chunks]
bm25_index = BM25Okapi(bm25_corpus)

print("FAISS vectors :", faiss_index.ntotal)
print("BM25 documents :", len(bm25_corpus))

## 11. Recherche hybride et test exploratoire

La fonction applique une fusion Reciprocal Rank Fusion (RRF). Les filtres `referentiel`, `language` et `level` sont facultatifs.

Les résultats doivent être évalués manuellement sur des questions connues avant de considérer le RAG comme validé.

In [ ]:
def hybrid_search(query, top_k=5, candidate_k=20, referentiel=None, language=None, level=None, rrf_k=60):
    q_emb = model.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    faiss_scores, faiss_ids = faiss_index.search(q_emb, min(candidate_k, len(chunks)))

    bm25_scores = bm25_index.get_scores(tokenize(query))
    bm25_ids = np.argsort(bm25_scores)[::-1][:candidate_k]

    fused = {}
    for rank, idx in enumerate(faiss_ids[0], start=1):
        if idx >= 0:
            fused[int(idx)] = fused.get(int(idx), 0) + 1 / (rrf_k + rank)
    for rank, idx in enumerate(bm25_ids, start=1):
        fused[int(idx)] = fused.get(int(idx), 0) + 1 / (rrf_k + rank)

    ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)
    results = []
    for idx, score in ranked:
        c = chunks[idx]
        if referentiel and c["referentiel"] != referentiel:
            continue
        if language and c["language"] != language:
            continue
        if level and c["level"] != level:
            continue
        results.append({"score_rrf": score, **c})
        if len(results) >= top_k:
            break
    return results

# Remplacer ces exemples par des questions métier préparées et validées.
TEST_QUERIES = [
    {"query": "grammatical structures for describing past events", "referentiel": "CECRL"},
    {"query": "汉语词汇和语法练习", "referentiel": "HSK"},
]

for test in TEST_QUERIES:
    print("\\nQUESTION :", test["query"])
    results = hybrid_search(test["query"], top_k=3, referentiel=test["referentiel"])
    for r in results:
        print(f"- {r['referentiel']} | {r['level']} | {r['source_file']} | {r['source_locator']} | RRF={r['score_rrf']:.4f}")
        print(r["text"][:350], "\\n")

## 12. Exporter le pack RAG

Le pack contient :
- `chunks.jsonl` : textes et métadonnées.
- `faiss.index` : index vectoriel.
- `bm25.pkl` : index BM25 sérialisé.
- `manifest.json` : modèle et paramètres.
- `quality_report.json` : statistiques d'extraction et d'indexation.

Le modèle d'embeddings n'est pas copié dans le pack : FastAPI devra le télécharger ou le charger depuis un cache local autorisé. L'index FAISS est dépendant de sa dimension et de son modèle d'embeddings.

In [ ]:
# Nettoyer uniquement le dossier d'export généré par ce notebook.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "chunks.jsonl", "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\\n")

faiss.write_index(faiss_index, str(OUTPUT_DIR / "faiss.index"))

with open(OUTPUT_DIR / "bm25.pkl", "wb") as f:
    pickle.dump(bm25_index, f, protocol=pickle.HIGHEST_PROTOCOL)

manifest = {
    "pack_name": "langue_ia_rag_cecrl_hsk",
    "pack_version": "0.1.0",
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": int(dimension),
    "embedding_normalized": True,
    "vector_index": "faiss.index",
    "lexical_index": "bm25.pkl",
    "chunks_file": "chunks.jsonl",
    "fusion": "reciprocal_rank_fusion",
    "rrf_k": 60,
    "chunk_size_chars": CHUNK_SIZE,
    "chunk_overlap_chars": CHUNK_OVERLAP,
    "referentials": ["CECRL", "HSK"],
    "audio_included": False,
    "created_with": "Google Colab",
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

quality_report = {
    "source_files": len(all_files),
    "json_files": len(json_files),
    "pdf_files": len(pdf_files),
    "pdf_pages": len(pdf_pages),
    "documents": len(documents),
    "chunks": len(chunks),
    "embedding_shape": list(embeddings.shape),
    "pdf_audit": pdf_audit,
    "document_counts": docs_df.groupby(["referentiel", "language"], dropna=False).size().reset_index(name="count").to_dict(orient="records"),
}
(OUTPUT_DIR / "quality_report.json").write_text(
    json.dumps(quality_report, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Pack exporté :", OUTPUT_DIR)
print([p.name for p in OUTPUT_DIR.iterdir()])

## 13. Télécharger le pack

Le ZIP exporté est le livrable technique à transmettre au développeur chargé de l'intégration FastAPI. Ne pas le placer directement dans Git si sa taille est importante : choisir un stockage de fichiers adapté au projet et configurer le chemin de chargement côté Service IA.

In [ ]:
EXPORT_ZIP = Path("/content/langue_ia_rag_pack.zip")
if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(OUTPUT_DIR)))

print(f"Archive prête : {EXPORT_ZIP} ({EXPORT_ZIP.stat().st_size / (1024**2):.1f} Mo)")

from google.colab import files
files.download(str(EXPORT_ZIP))

## 14. Vérifications et limites avant intégration

Avant de remettre le pack au développeur :
- vérifier les exemples de chunks CECRL et HSK ;
- vérifier que les niveaux et langues ne sont pas incorrectement détectés ;
- examiner les pages HSK signalées comme nécessitant un OCR ;
- préparer un jeu de questions avec les documents attendus ;
- comparer BM25 seul, vectoriel seul et hybride sur ce jeu ;
- vérifier les droits et conditions d'utilisation du modèle d'embeddings ;
- faire valider le format du pack par le développeur FastAPI.

### Limites de cette version
1. L'extraction PDF ne fait pas d'OCR.
2. Les champs JSON sont détectés par heuristiques : chaque schéma doit être vérifié.
3. La tokenisation BM25 est volontairement simple, notamment pour le chinois.
4. Le score RRF n'est pas une probabilité de pertinence.
5. Ce notebook produit un index expérimental : il ne remplace pas les tests d'intégration et d'évaluation pédagogique.
6. Le pickle BM25 doit être chargé uniquement depuis une source de confiance.